<a href="https://colab.research.google.com/github/lqi234488/AICUP-2026-Table-Tennis-Prediction/blob/main/notebooks/%E5%9F%BA%E6%96%BC%E6%99%82%E5%BA%8F%E8%B3%87%E6%96%99%E4%B9%8B%E6%A1%8C%E7%90%83%E6%88%B0%E8%A1%93%E8%88%87%E7%B5%90%E6%9E%9C%E9%A0%90%E6%B8%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

基於時序資料之桌球戰術與結果預測競賽

In [ ]:
import pandas as pd
import numpy as np

# 1. 讀取資料
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# 2. 核心特徵選取
# 這些是每一拍會變動的特徵 (Time-step features)
features = ['strikeId', 'handId', 'strengthId', 'spinId', 'positionId', 'actionId']
target = ['actionId','pointId','serverGetPoint'] # 假設我們要預測下一拍的動作

# 3. 依照 rally_uid 進行分組轉換
def create_sequences(df, max_len=15):
    sequences = []
    labels = []

    for uid, group in df.groupby('rally_uid'):
        # 確保依擊球順序排列
        group = group.sort_values('strikeNumber')

        # 提取特徵矩陣
        feat_matrix = group[features].values

        # 建立滑動窗口 (例如用前 3 拍預測第 4 拍)
        for i in range(1, len(feat_matrix)):
            seq = feat_matrix[:i] # 前 i 拍
            # 補零到固定長度 max_len
            if len(seq) < max_len:
                pad = np.zeros((max_len - len(seq), len(features)))
                seq = np.vstack((pad, seq))
            else:
                seq = seq[-max_len:]

            sequences.append(seq)
            labels.append(group.iloc[i][target]) # 第 i+1 拍的動作

    return np.array(sequences), np.array(labels)

# 執行轉換 (這步會花一點時間)
# X_train, y_train = create_sequences(train_df)

檢查

In [ ]:
import pandas as pd
import os

print("目前目錄下的檔案有：", os.listdir('/content/'))
# 1. 自動檢查 Colab 目前有哪些檔案
current_files = os.listdir('/content/')
print(f"目前目錄下的檔案：{current_files}")

# 2. 嘗試用正確的路徑讀取 (加上 /content/ 前綴)
try:
    # 這裡確保路徑是 Colab 的預設路徑
    train = pd.read_csv('/content/train.csv')
    print("✅ 成功讀取 train.csv！")

    # 3. 執行你剛才想跑的分析
    print("\n--- 每一個回合(Rally)的拍數統計 ---")
    strike_counts = train.groupby('rally_uid')['strikeNumber'].max()
    print(strike_counts.value_counts().sort_index())

except FileNotFoundError:
    print("❌ 錯誤：找不到檔案！請確認左側資料夾選單中，檔名是否真的叫 'train.csv'")
except Exception as e:
    print(f"❌ 發生其他錯誤：{e}")

In [ ]:
# 範例：將資料按 rally 排序
train = train.sort_values(['rally_uid', 'strikeNumber'])

# 找出所有的類別數量 (用於 Embedding)
num_actions = train['actionId'].nunique()
num_positions = train['positionId'].nunique()
num_getpoint = train['serverGetPoint'].nunique()

print(f"總共有 {num_actions} 種動作，{num_positions} 種落點方位。{num_getpoint} 種得分")

前處理

In [49]:
import pandas as pd

def engineering_features(df):
    # 1. 數值型歸一化 (讓數值落在 0~1 之間，對 LSTM 收斂很有幫助)
    df['strengthId'] = df['strengthId'] / 3.0
    df['scoreSelf'] = df['scoreSelf'] / 15.0
    df['scoreOther'] = df['scoreOther'] / 15.0
    df['strikeNumber'] = df['strikeNumber'] / 52.0 # 假設一回合最多30拍

    # 2. 類別型 One-hot Encoding
    # 我們只針對純類別進行展開
    df = pd.get_dummies(df, columns=['handId', 'spinId', 'positionId'])

    return df

# 執行轉換
train_df_eng = engineering_features(train)
test_df_eng = engineering_features(test)

# 【重要】對齊測試集與訓練集的欄位 (避免 One-hot 後欄位數量不一致)
train_df_eng, test_df_eng = train_df_eng.align(test_df_eng, join='left', axis=1, fill_value=0)

# 找出現在所有的特徵欄位名稱 (扣除標籤和 UID)
# 排除掉那三個 Output 標籤
exclude_cols = ['rally_uid', 'actionId', 'pointId', 'serverGetPoint']
NEW_FEATURE_COLS = [c for c in train_df_eng.columns if c not in exclude_cols]

print(f"新的特徵數量：{len(NEW_FEATURE_COLS)}")

新的特徵數量：24


第一步：prepare_data 構建訓練資料 (X, y)
我們需要把 train.csv 的每一列轉成「序列」，並提取每個 rally_uid 的最終結果作為標籤。

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 設定參數
MAX_SEQ_LEN = 10  # 每個回合考慮前 10 拍，不足補 0
FEATURE_COLS = ['strikeNumber', 'scoreSelf', 'scoreOther', 'handId', 'spinId', 'positionId', 'serverGetPoint']

def prepare_data_v5(df, max_len=10):
    _X, _y_action, _y_point, _y_server, _uids = [], [], [], [], []
    # 根據你的想法剔除不具參考性的特徵 (例如 sex, match)
    feature_cols = [
        'strikeNumber', 'scoreSelf', 'scoreOther', 'serverGetPoint',
        'handId', 'strengthId', 'spinId', 'positionId'
    ]

    for uid, group in df.groupby('rally_uid'):
        group = group.sort_values('strikeNumber')
        feat_matrix = group[feature_cols].values
        for i in range(1, len(feat_matrix)):
            seq = feat_matrix[:i]
            if len(seq) < max_len:
                pad = np.zeros((max_len - len(seq), len(feature_cols)))
                seq = np.vstack((pad, seq))
            else:
                seq = seq[-max_len:]

            _X.append(seq)
            row = group.iloc[i]
            _y_action.append(row['actionId'])
            _y_point.append(row['pointId'])
            _y_server.append(row['serverGetPoint'])
            _uids.append(uid) # 記錄這組資料是屬於哪個 rally

    return (np.array(_X), np.array(_y_action).astype('int32'), np.array(_y_point).astype('int32'), np.array(_y_server).astype('float32'),_uids)

# 執行讀取
X_train, y_action, y_point, y_server, train_uids = prepare_data_v5(train)
X_test, _, _, _, test_uids = prepare_data_v5(test)
print(f"新的訓練樣本數: {len(X_train)}, {len(y_action)}, {len(y_point)}, {len(y_server)}, {len(train_uids)}")
print(f"修正後的 X 形狀: {X_train.shape}")
print(f"新的測試樣本數: {len(X_test)}, {len(test_uids)}")
print(f"修正後的 X 形狀: {X_test.shape}")

第二步：建立多輸出模型 (Keras Functional API)
這段程式碼會建立一個 LSTM 模型，最後分出三個輸出層：

In [ ]:
from re import X
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split

num_classes = 10

# 重新定義模型
inputs = Input(shape=(10, 8))
x = LSTM(64, return_sequences=True)(inputs)
x = Dropout(0.2)(x)
x = LSTM(32)(x)
x = Dense(64, activation='relu')(x)
# 最終輸出：第 N 球的落點機率
outputs = Dense(num_classes, activation='softmax')(x)

# --- 關鍵：明確命名輸出層 ---
# Action (動作0-18 -> 19個輸出)
out_action = Dense(17, activation='softmax', name='act_head')(x)
# Point (落點 0-9 -> 10個輸出)
out_point = Dense(11, activation='softmax', name='ptr_head')(x)
# Server (是否得分 0-1 -> 1個輸出)
out_server = Dense(1, activation='sigmoid', name='srv_head')(x)

model_multi = Model(inputs=inputs, outputs=[out_action, out_point, out_server])

model_multi.compile(
    optimizer='adam',
    loss={
        'act_head': 'sparse_categorical_crossentropy',
        'ptr_head': 'sparse_categorical_crossentropy',
        'srv_head': 'binary_crossentropy'
    },
    # 這裡也要改成字典，對應三個輸出的名稱
    metrics={
        'act_head': 'accuracy',
        'ptr_head': 'accuracy',
        'srv_head': 'accuracy'
    },
    loss_weights={'act_head': 1.0, 'ptr_head': 1.2, 'srv_head': 0.8}
)

# 1. 手動切分資料 (解決 NameError)
# 我們把 prepare_data_v5 產生的 X_train, y_act, y_ptr, y_srv 進行切分
X_train, X_val, \
y_act_train, y_act_val, \
y_ptr_train, y_ptr_val, \
y_srv_train, y_srv_val = train_test_split(
    X_train, y_action, y_point, y_server,
    test_size=0.2,    # 撥出 20% 作為驗證集 (模擬考)
    random_state=42   # 固定隨機種子，確保每次切分結果一樣
)

print(f"✅ 資料切分成功！")
print(f"訓練樣本數: {len(X_train)}")
print(f"驗證樣本數: {len(X_val)}")





# 使用字典匹配名稱，並傳入手動切好的驗證集
history = model_multi.fit(
    X_train,
    {
        'act_head': y_act_train,
        'ptr_head': y_ptr_train,
        'srv_head': y_srv_train
    },
    validation_data=(
        X_val,
        {
            'act_head': y_act_val,
            'ptr_head': y_ptr_val,
            'srv_head': y_srv_val
        }
    ),
    epochs=50,
    batch_size=64,
    verbose=1
)

model_multi.summary()

進行預測

In [ ]:
import numpy as np
import pandas as pd

# 1. 進行預測 (會得到三個輸出的機率分佈)
predictions = model_multi.predict(X_test)

# 2. 轉換結果：將機率轉為具體的類別 ID
# 假設你的輸出順序是 [act_head, ptr_head, srv_head]
pred_actions = np.argmax(predictions[0], axis=1) # 擊球動作 (0-16)
pred_points = np.argmax(predictions[1], axis=1)  # 落點區域 (0-10)
pred_servers = (predictions[2] > 0.5).astype(int).flatten() # 是否得分 (0 或 1)

# 3. 檢查預測出的維度是否為 2353
print(f"預測完成！樣本數：{len(pred_actions)}")

建立並匯出 Submission 檔案

In [ ]:
# 建立符合競賽格式的 DataFrame
submission = pd.DataFrame({
    'rally_uid': test_uids,
    'actionId': pred_actions,
    'pointId': pred_points,
    'serverGetPoint': pred_servers
})

# 儲存為 CSV
file_name = 'submission_v1.csv'
submission.to_csv(file_name, index=False)

print(f"✅ 檔案已儲存為 {file_name}")

# 如果在 Colab，執行這行直接下載到電腦
try:
    from google.colab import files
    files.download(file_name)
except:
    print("非 Colab 環境，請手動從資料夾下載檔案。")

In [ ]:
# 看看模型預測的落點是不是都集中在某幾區
print("落點預測分佈：")
print(submission['pointId'].value_counts().sort_index())